# Intent Classification — n8n Batch Runner

Sends each row of `classification_test_case.xlsx` to the **Intent Classification** webhook and appends `result_route_to` and `result_narrative` columns.
Output is saved to `classification_result.xlsx`.

In [9]:
# ── Configuration ──────────────────────────────────────────────────────────────
N8N_BASE_URL  = "https://alphamakeathon-automation.arisetech.dev"
WEBHOOK_PATH  = "dbce5b9e-1397-459a-871a-5b27433f1640"
USE_TEST_URL  = False   # True → /webhook-test/…  |  False → /webhook/…

INPUT_FILE    = "classification_test_case.xlsx"
OUTPUT_FILE   = "classification_result.xlsx"

TIMEOUT       = 30      # seconds per request
RETRIES       = 2       # extra attempts on failure
DELAY_BETWEEN = 0.2     # seconds between rows (avoid rate-limiting)

In [10]:
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

In [11]:
def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


def call_intent_classifier(row: dict, timeout: int = TIMEOUT, retries: int = RETRIES) -> dict:
    payload = {
        "userMessage":   str(row.get("userMessage",   "") or ""),
        "LastAImessage": str(row.get("LastAImessage", "") or ""),
        "PrevAIagent":   str(row.get("PrevAIagent",   "") or ""),
        "narrative":     str(row.get("narrative",     "") or ""),
    }
    url = get_webhook_url()
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last_exc


print("Webhook URL:", get_webhook_url())

Webhook URL: https://alphamakeathon-automation.arisetech.dev/webhook/dbce5b9e-1397-459a-871a-5b27433f1640


In [12]:
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df)} rows  |  columns: {list(df.columns)}")
df.head(3)

Loaded 250 rows  |  columns: ['testId', 'userMessage', 'LastAImessage', 'PrevAIagent', 'narrative', 'expected_route', 'reason']


,testId,userMessage,LastAImessage,PrevAIagent,narrative,expected_route,reason
0,TC-0001,ตอนนี้โบนัสถูกเลื่อน เลยอยากถามว่าอยากลดค่างวด...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,Customer raises a personal debt-management sit...,advisor,Customer requests a personalized debt solution...
1,TC-0002,ถ้าสถานการณ์คือโบนัสถูกเลื่อน แบบนี้ คุณช่วยแน...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,Customer raises a personal debt-management sit...,advisor,Customer requests a personalized debt solution...
2,TC-0003,แผนที่ให้มาดูดีนะคะ แต่พอดีโบนัสถูกเลื่อน เลยอ...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,Customer raises a personal debt-management sit...,advisor,Customer requests a personalized debt solution...


In [13]:
result_route_to  = []
result_narrative = []
errors           = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Calling n8n"):
    try:
        resp = call_intent_classifier(row.to_dict())
        result_route_to.append(resp.get("route_to", ""))
        result_narrative.append(resp.get("narrative", ""))
        errors.append(None)
    except Exception as exc:
        result_route_to.append(None)
        result_narrative.append(None)
        errors.append(str(exc))
    time.sleep(DELAY_BETWEEN)

df["result_route_to"]  = result_route_to
df["result_narrative"] = result_narrative
df["error"]            = errors

failed = df["error"].notna().sum()
print(f"Done. {len(df) - failed}/{len(df)} succeeded, {failed} failed.")

Calling n8n:   0%|          | 0/250 [00:00<?, ?it/s]

Done. 250/250 succeeded, 0 failed.


In [14]:
# Preview failed rows (if any)
if failed:
    display(df[df["error"].notna()][["testId", "userMessage", "error"]])

In [15]:
df.to_excel(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
df[["testId", "expected_route", "result_route_to", "result_narrative"]].head(5)

Saved → classification_result.xlsx


,testId,expected_route,result_route_to,result_narrative
0,TC-0001,advisor,advisor,The customer states their bonus has been postp...
1,TC-0002,advisor,advisor,The advisor proposed a restructuring plan. The...
2,TC-0003,advisor,advisor,The advisor proposed a debt plan. The customer...
3,TC-0004,advisor,advisor,The customer states their bonus has been postp...
4,TC-0005,advisor,advisor,The customer states their bonus has been delay...
